In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})

BLUE = "#006DAE"
ORANGE = "#E57200"
GREEN = "#00843D"
RED = "#C8102E"
PURPLE = "#7A52A1"

# Conditions for Blasius
INITIAL_F = 0.0
INITIAL_dF = 0.0
FINAL_dF = 1.0
INITIAL_SUBMERGE_SCALED = 0.0
MAX_SUBMERGE_SCALED = 0.0
GUESS_ddF = 1.0
GUESS2_ddF = 20.0

STEP_SIZE = 0.1

# Blasius ODE initial condition
def blasius_rhs(submerge_scaled, F, dF, ddF):
    # F is the scaled stream function
    # ddf is the scaled vecloty stream function
    # dF is u/U, i.e., velocity along the board as a fraction of the free-stream speed

    '''Return dT/dt [K/s] for the lumped thermal model.'''
    return np.array([
        dF,
        ddF,
        (1/2)*F*ddF
    ])
print(blasius_rhs(INITIAL_SUBMERGE_SCALED, 0.3, 0.6, 0.2))

def thermal_exact(time_s, initial_c=INITIAL_C, power_w=HEATER_W,
                  capacitance_j_k=C_J_K, conductance_w_k=H_W_K,
                  ambient_c=AMBIENT_C, initial_time_s=0.0):
    '''Exact constant-input thermal trajectory used for validation.'''
    time_s = np.asarray(time_s, dtype=float)
    equilibrium_c = ambient_c + power_w/conductance_w_k
    tau_s = capacitance_j_k/conductance_w_k
    return equilibrium_c + (initial_c-equilibrium_c)*np.exp(
        -(time_s-initial_time_s)/tau_s
    )

print(f"Thermal time constant = {TAU_S:.1f} s; "
      f"equilibrium = {EQUILIBRIUM_C:.1f} °C")


[0.6  0.2  0.03]
Thermal time constant = 240.0 s; equilibrium = 42.0 °C


In [ ]:
k1 = thermal_rhs(t0, T0)
T_trial_2 = T0 + 0.5*h_demo*k1
k2 = thermal_rhs(t0+0.5*h_demo, T_trial_2)
T_trial_3 = T0 + 0.5*h_demo*k2
k3 = thermal_rhs(t0+0.5*h_demo, T_trial_3)
T_trial_4 = T0 + h_demo*k3
k4 = thermal_rhs(t0+h_demo, T_trial_4)
weighted_rate = (k1+2*k2+2*k3+k4)/6
T_rk4_next = T0+h_demo*weighted_rate

print(" stage   trial time [s]   trial T [°C]   rate k [K/s]")
stage_rows = [
    (1, t0, T0, k1),
    (2, t0+h_demo/2, T_trial_2, k2),
    (3, t0+h_demo/2, T_trial_3, k3),
    (4, t0+h_demo, T_trial_4, k4),
]
for stage, trial_time, trial_temperature, rate in stage_rows:
    print(f" {stage:3d}   {trial_time:14.1f}   {trial_temperature:12.4f}   {rate:13.6f}")
print(f"\nWeighted rate = {weighted_rate:.6f} K/s")
print(f"RK4 next state = {T_rk4_next:.5f} °C")
print(f"Exact next state = {T_exact_next:.5f} °C")

plt.figure(figsize=(9, 5))
plt.plot(local_time, exact_local, color="black", linewidth=2,
         label="Exact trajectory")
stage_times = [row[1] for row in stage_rows]
stage_temperatures = [row[2] for row in stage_rows]
plt.scatter(stage_times, stage_temperatures, c=[BLUE, ORANGE, PURPLE, RED],
            s=85, zorder=4, label="RK4 trial states")
for stage, trial_time, trial_temperature, _ in stage_rows:
    plt.annotate(f"$k_{stage}$", (trial_time, trial_temperature),
                 xytext=(6, 7), textcoords="offset points")
plt.scatter(h_demo, T_rk4_next, marker="s", color=GREEN, s=80,
            label="Accepted RK4 next state", zorder=5)
plt.xlabel("Time [s]")
plt.ylabel("Temperature [°C]")
plt.title("RK4 trial states are slope-sampling locations")
plt.legend()
plt.tight_layout()
plt.show()
